# Reinforcement Learning Basics

## Learning Objectives
- Understand key concepts of reinforcement learning (RL).
- Learn differences between RL and supervised learning.
- Learn simple RL algorithms and components.

## Introduction
Reinforcement Learning (RL) algorithms help the computer to make a sequence of decissions. The algorithm learns to achieve in an uncertain, potentially complex environment.

Here an agent makes decisions by following a policy based on which actions to take, and it learns from the consequences of these actions through rewards or penalties

## Common types of Reinforcement Learning tasks

Markov Decission Process for how the state of a system evolves as different actions are applied to the system. A few different quantities come together to form an MDP

Q-learning: This is a model-free reinforcement learning algorithm that learns the value of an action in a particular state.

Deep Q-Networks (DQN): It combines Q-learning with deep neural networks, allowing the approach to learn successful policies directly from high-dimensional sensory inputs.

Policy Gradient Methods: These methods optimize the parameters of a policy directly as opposed to estimating the value of actions.

Monte Carlo Tree Search (MCTS): Used in decision processes for finding optimal decisions by playing out scenarios, notably used in games like Go.

SARSA (State-Action-Reward-State-Action): An on-policy learning algorithm. Uses the actual reward of the next action taken (current policy) for updates. Follows the current policy, considering exploration during updates.

## Core Concepts

- Agent and Environment: These are the two primary components in Reinforcement Learning.
    - The Agent is the learner or decision-maker (e.g., a game-playing AI).
    - The Environment is the world the agent interacts with (e.g., the game itself). The interaction is a loop: the agent performs an action, the environment transitions to a new state, and the agent receives a reward.

- Reward Signal: This is a numerical value the environment gives the agent after each action. It provides immediate feedback: positive rewards for desirable actions and negative rewards (or punishments) for undesirable ones. The agent's sole objective is to maximize the total cumulative reward it receives over time.

- Policy: The policy is the agent's strategy or "brain." It defines the agent's behavior by mapping states to actions. In other words, it's the rule that tells the agent what action to take when it's in a particular state. The goal of RL is to find the optimal policy that yields the most reward.

- Value Function: While the reward signal provides immediate feedback, the value function estimates the long-term desirability of a state. It represents the total amount of reward an agent can expect to accumulate in the future, starting from that state. It helps the agent make decisions that are good in the long run, not just for the immediate next step.

- Exploration vs. Exploitation: This is the fundamental trade-off in RL.
    - Exploitation: The agent uses its current knowledge to choose the action it believes will give the best reward.
    - Exploration: The agent tries a new or random action to discover more about the environment. The agent must balance exploring to find new, potentially better strategies with exploiting its current best strategy to maximize rewards.


In [1]:
# Example: Bandit problem simulation
import numpy as np

true_rewards = [0.2, 0.5, 0.75]  # Probabilities of reward for 3 actions
chosen_action = 1  # Choose second action
reward = 1 if np.random.rand() < true_rewards[chosen_action] else 0
print(f"Reward received: {reward}")

Reward received: 1


A more robust example

In [2]:
# Simple epsilon-greedy bandit algorithm
num_actions = len(true_rewards)
num_iterations = 1000
epsilon = 0.1

# Initialize estimates and counts
Q = np.zeros(num_actions)  # Action value estimates
N = np.zeros(num_actions)  # Number of times each action was chosen

rewards_history = []

for t in range(num_iterations):
    # Epsilon-greedy action selection
    if np.random.random() < epsilon:
        action = np.random.randint(num_actions)  # Explore
    else:
        action = np.argmax(Q)  # Exploit
    
    # Get reward
    reward = 1 if np.random.random() < true_rewards[action] else 0
    rewards_history.append(reward)
    
    # Update estimates
    N[action] += 1
    Q[action] = Q[action] + (1/N[action]) * (reward - Q[action])

# Print results
print("Estimated action values:", Q)
print("True action values:", true_rewards)
print("Number of times each action was chosen:", N)
print("Average reward:", np.mean(rewards_history))

Estimated action values: [0.2        0.56       0.74915254]
True action values: [0.2, 0.5, 0.75]
Number of times each action was chosen: [ 65.  50. 885.]
Average reward: 0.704


Agent example

In [ ]:
# %pip install gym
# %pip install pyglet==1.2.4
# %pip install pygame

In [ ]:
import gymnasium as gym
import numpy as np
import math

# Create the CartPole environment
env = gym.make('CartPole-v1')

# --- Hyperparameters ---
EPISODES = 20000
LEARNING_RATE = 0.1
GAMMA = 0.99
# Exploration parameters
EPSILON_START = 1.0
EPSILON_END = 0.01
EPSILON_DECAY_RATE = 0.99995

# --- Discretization ---
# Number of bins for each observation dimension
# (Cart Position, Cart Velocity, Pole Angle, Pole Velocity)
n_bins = (10, 10, 10, 10)

# Observation space bounds. Velocities are technically infinite,
# so we set reasonable bounds based on common practice for this problem.
obs_space_bounds = [
    (-4.8, 4.8),
    (-4.0, 4.0),
    (-0.418, 0.418), # ~24 degrees
    (-4.0, 4.0)
]

# Q-table size
q_table_size = n_bins + (env.action_space.n,)
q_table = np.zeros(q_table_size)

def discretize_state(state):
    """Converts a continuous state into a discrete tuple which acts as an index."""
    discrete_state = []
    for i, s in enumerate(state):
        low, high = obs_space_bounds[i]
        # Clip the value to ensure it's within our defined bounds
        s = np.clip(s, low, high)
        # Calculate the bin index for the current state dimension
        bin_index = int(((s - low) / (high - low)) * (n_bins[i] - 1))
        discrete_state.append(bin_index)
    return tuple(discrete_state)

# --- Q-learning Agent ---
class QLearningAgent:
    def __init__(self):
        self.epsilon = EPSILON_START

    def get_action(self, state):
        """Choose an action using an epsilon-greedy strategy."""
        if np.random.random() < self.epsilon:
            return env.action_space.sample()  # Explore: choose a random action
        return np.argmax(q_table[state])  # Exploit: choose the best known action

    def learn(self, state, action, reward, next_state):
        """Update Q-value for a given state-action pair using the Bellman equation."""
        old_value = q_table[state + (action,)]
        next_max = np.max(q_table[next_state])
        
        # Q-learning formula
        new_value = (1 - LEARNING_RATE) * old_value + LEARNING_RATE * (reward + GAMMA * next_max)
        q_table[state + (action,)] = new_value

    def decay_epsilon(self):
        """Decay epsilon to reduce exploration and increase exploitation over time."""
        if self.epsilon > EPSILON_END:
            self.epsilon *= EPSILON_DECAY_RATE

# --- Training Loop ---
agent = QLearningAgent()
scores = []

print("Training started...")
for episode in range(EPISODES):
    # Reset environment and get initial discrete state
    state, info = env.reset()
    state = discretize_state(state)
    
    total_reward = 0
    terminated = False
    truncated = False
    
    while not terminated and not truncated:
        action = agent.get_action(state)
        
        # Take action and get feedback from the environment
        next_state_continuous, reward, terminated, truncated, _ = env.step(action)
        next_state = discretize_state(next_state_continuous)
        
        # Penalize the agent for failing (the pole falling over)
        if terminated:
            reward = -20

        agent.learn(state, action, reward, next_state)
        
        state = next_state
        total_reward += reward
    
    # Decay epsilon after each episode
    agent.decay_epsilon()
    
    scores.append(total_reward)
    if (episode + 1) % 1000 == 0:
        avg_score = np.mean(scores[-1000:])
        print(f'Episode {episode + 1} | Avg Score (last 1000): {avg_score:.2f} | Epsilon: {agent.epsilon:.4f}')

env.close()
print(f'\nTraining finished.\nAverage Score over all episodes: {np.mean(scores):.2f}')


Training started...
Episode 1000 | Avg Score (last 1000): 2.33 | Epsilon: 0.9512
Episode 2000 | Avg Score (last 1000): 3.85 | Epsilon: 0.9048
Episode 3000 | Avg Score (last 1000): 6.64 | Epsilon: 0.8607
Episode 4000 | Avg Score (last 1000): 9.57 | Epsilon: 0.8187
Episode 5000 | Avg Score (last 1000): 12.87 | Epsilon: 0.7788
Episode 6000 | Avg Score (last 1000): 15.14 | Epsilon: 0.7408
Episode 7000 | Avg Score (last 1000): 20.85 | Epsilon: 0.7047
Episode 8000 | Avg Score (last 1000): 22.90 | Epsilon: 0.6703
Episode 9000 | Avg Score (last 1000): 27.63 | Epsilon: 0.6376
Episode 10000 | Avg Score (last 1000): 32.87 | Epsilon: 0.6065
Episode 11000 | Avg Score (last 1000): 35.82 | Epsilon: 0.5769
Episode 12000 | Avg Score (last 1000): 39.75 | Epsilon: 0.5488
Episode 13000 | Avg Score (last 1000): 45.48 | Epsilon: 0.5220
Episode 14000 | Avg Score (last 1000): 50.88 | Epsilon: 0.4966
Episode 15000 | Avg Score (last 1000): 55.26 | Epsilon: 0.4724
Episode 16000 | Avg Score (last 1000): 61.84 | E

Policy-based example

In [15]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical

# --- Hyperparameters ---
LEARNING_RATE = 0.01
GAMMA = 0.99  # Discount factor for future rewards
EPISODES = 2000
LOG_INTERVAL = 100 # How often to print progress

# --- Policy Network ---
# A simple neural network that takes a state and outputs action probabilities.
# This network *is* the policy.
class PolicyNetwork(nn.Module):
    def __init__(self, state_size, action_size):
        super(PolicyNetwork, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(state_size, 128),
            nn.ReLU(),
            nn.Linear(128, action_size),
            nn.Softmax(dim=-1)  # Outputs a probability distribution over actions
        )

    def forward(self, state):
        return self.layers(state)

# --- REINFORCE Agent ---
class ReinforceAgent:
    def __init__(self, state_size, action_size):
        self.policy_network = PolicyNetwork(state_size, action_size)
        self.optimizer = optim.Adam(self.policy_network.parameters(), lr=LEARNING_RATE)
        
        # Store rewards and log probabilities for the current episode
        self.rewards = []
        self.log_probs = []

    def get_action(self, state):
        """
        Selects an action by sampling from the policy network's output distribution.
        """
        state_tensor = torch.from_numpy(state).float().unsqueeze(0)
        action_probs = self.policy_network(state_tensor)
        
        # Create a probability distribution and sample an action
        dist = Categorical(action_probs)
        action = dist.sample()
        
        # Store the log probability of the chosen action (needed for the loss calculation)
        self.log_probs.append(dist.log_prob(action))
        
        return action.item()

    def learn(self):
        """
        Updates the policy network based on the rewards received in an episode.
        This is the core of the REINFORCE algorithm.
        """
        policy_loss = []
        discounted_rewards = []
        
        # Calculate discounted rewards for the episode (from the end to the beginning)
        R = 0
        for r in reversed(self.rewards):
            R = r + GAMMA * R
            discounted_rewards.insert(0, R)
            
        # Normalize rewards for more stable training
        discounted_rewards = torch.tensor(discounted_rewards)
        discounted_rewards = (discounted_rewards - discounted_rewards.mean()) / (discounted_rewards.std() + 1e-9)

        # Calculate the policy loss
        for log_prob, R in zip(self.log_probs, discounted_rewards):
            # The loss is the negative log probability of the action multiplied by the reward.
            # Maximizing reward is equivalent to minimizing this loss.
            policy_loss.append(-log_prob * R)

        # Update the network weights
        self.optimizer.zero_grad()
        policy_loss = torch.cat(policy_loss).sum()
        policy_loss.backward()
        self.optimizer.step()
        
        # Clear the episode's data
        self.rewards = []
        self.log_probs = []


# --- Training Loop ---
env = gym.make('CartPole-v1')
state_size = env.observation_space.shape[0]
action_size = env.action_space.n

agent = ReinforceAgent(state_size, action_size)
scores = []

print("Training started...")
for episode in range(EPISODES):
    state, info = env.reset()
    total_reward = 0
    
    terminated = False
    truncated = False

    while not terminated and not truncated:
        action = agent.get_action(state)
        state, reward, terminated, truncated, _ = env.step(action)
        
        agent.rewards.append(reward)
        total_reward += reward
        
    # At the end of the episode, update the policy
    agent.learn()
    
    scores.append(total_reward)
    if (episode + 1) % LOG_INTERVAL == 0:
        avg_score = np.mean(scores[-LOG_INTERVAL:])
        print(f'Episode {episode + 1} | Average Score (last {LOG_INTERVAL}): {avg_score:.2f}')

env.close()
print(f'\nTraining finished.\nFinal Average Score: {np.mean(scores):.2f}')

Training started...
Episode 100 | Average Score (last 100): 144.16
Episode 200 | Average Score (last 100): 103.80
Episode 300 | Average Score (last 100): 497.32
Episode 400 | Average Score (last 100): 500.00
Episode 500 | Average Score (last 100): 273.22
Episode 600 | Average Score (last 100): 102.54
Episode 700 | Average Score (last 100): 68.80
Episode 800 | Average Score (last 100): 68.05
Episode 900 | Average Score (last 100): 138.85
Episode 1000 | Average Score (last 100): 146.55
Episode 1100 | Average Score (last 100): 81.54
Episode 1200 | Average Score (last 100): 166.46
Episode 1300 | Average Score (last 100): 55.14
Episode 1400 | Average Score (last 100): 125.72
Episode 1500 | Average Score (last 100): 254.40
Episode 1600 | Average Score (last 100): 409.34
Episode 1700 | Average Score (last 100): 500.00
Episode 1800 | Average Score (last 100): 500.00
Episode 1900 | Average Score (last 100): 492.00
Episode 2000 | Average Score (last 100): 351.98

Training finished.
Final Average

Markov Chains

In [8]:
np.random.seed(42)

transition_probabilities = [ # shape=[s, s']
        [0.7, 0.2, 0.0, 0.1],  # from s0 to s0, s1, s2, s3
        [0.0, 0.0, 0.9, 0.1],  # from s1 to ...
        [0.0, 1.0, 0.0, 0.0],  # from s2 to ...
        [0.0, 0.0, 0.0, 1.0]]  # from s3 to ...

n_max_steps = 50

def print_sequence():
    current_state = 0
    print("States:", end=" ")
    for step in range(n_max_steps):
        print(current_state, end=" ")
        if current_state == 3:
            break
        current_state = np.random.choice(range(4), p=transition_probabilities[current_state])
    else:
        print("...", end="")
    print()

for _ in range(10):
    print_sequence()

States: 0 0 3 
States: 0 1 2 1 2 1 2 1 2 1 3 
States: 0 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 3 
States: 0 3 
States: 0 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 3 
States: 0 1 3 
States: 0 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 ...
States: 0 0 3 
States: 0 0 0 1 2 1 2 1 3 
States: 0 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 3 


## Exercise
Simulate a simple epsilon-greedy strategy for the bandit problem.

In [ ]:
# Your code here

## Summary
- RL trains agents through interaction and feedback.
- Trade-off between exploration and exploitation is fundamental.


## Further Reading
- [Reinforcement Learning - Sutton & Barto](http://incompleteideas.net/book/the-book-2nd.html)
- [RL Course by David Silver](https://www.davidsilver.uk/teaching/)
